In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.backends.cudnn as cudnn
import numpy as np
from torchvision import transforms
import torchvision
import matplotlib.pyplot as plt
import time
import os
from PIL import Image
from tempfile import TemporaryDirectory
cudnn.benchmark = True
plt.ion()


transform = transforms.Compose([
    # Resize each image to 256x256
    transforms.Resize((256,256)),
    # Create a matrix representation of the image
    transforms.ToTensor(),
])


In [ ]:
# Set the number of images we're training on simultaneously
batch_size = 64

# Specify the data that we will train the model on and transform them into tensors
# We used ~84% of all our data for training
trainset = torchvision.datasets.ImageFolder(root='../train', transform=transform)
# Load the tensors using PyTorch's DataLoader (Reference #1)
# Enable shuffle so the model is given a variety of cards as it's training
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

# Specify the classes that the model will use to predict cards
classes = ( 
    '01', '02', '03', '04', '05', '06', '07', '08', '09', '10',
    '11', '12', '13', '14', '15', '16', '17', '18', '19', '20',
    '21', '22', '23', '24', '25', '26', '27', '28', '29', '30',
    '31', '32', '33', '34', '35', '36', '37', '38', '39', '40',
    '41', '42', '43', '44', '45', '46', '47', '48', '49', '50',
    '51', '52'
)


In [ ]:
# Repeat the process for the validation (test) data
# We used ~16% of all our data for validation
valset = torchvision.datasets.ImageFolder(root='../test', transform=transform)
valloader = torch.utils.data.DataLoader(valset, batch_size=batch_size,
                                        shuffle=False, num_workers=2)

print(f'Validation set size: {len(valset)} images')
print(f'Training set size: {len(trainset)} images')


Validation set size: 520 images
Training set size: 2758 images


In [ ]:
# PyTorch provides a Module method that serves as a backbone for custom networks (Reference #2)
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=52):
        super(SimpleCNN, self).__init__()

        self.features = nn.Sequential(
            # Convolutional layers derived from this article: (Reference #5)
            # The model uses 4 convolutional layers with ReLU and MaxPooling
            # Each layer halves the tensor size, and the layers themselves are connected in a loop
            nn.Conv2d(3, 32, kernel_size=3, padding=1),  # Input: 3x256x256, Output: 32x256x256
            nn.ReLU(),                                   # Activation function
            nn.MaxPool2d(2),                             # Max pooling, halves HxW -> 32x128x128

            nn.Conv2d(32, 64, kernel_size=3, padding=1), # Input: 32x128x128, Output: 64x128x128
            nn.ReLU(),
            nn.MaxPool2d(2),                             # Halve HxW -> 64x64x64

            nn.Conv2d(64, 128, kernel_size=3, padding=1),# Input: 64x64x64, Output: 128x64x64
            nn.ReLU(),
            nn.MaxPool2d(2),                             # Halve HxW -> 128x32x32

            nn.Conv2d(128, 256, kernel_size=3, padding=1),# Input: 128x32x32, Output: 256x32x32
            nn.ReLU(),
            nn.MaxPool2d(2),                             # Halve HxW -> 256x16x16
        )
        #Classifier
        self.classifier = nn.Sequential(
            # This is the final output from the model's last layer
            nn.Linear(256 * 16 * 16, 512),   # Flatten feature map and map to 512 neurons
            nn.ReLU(),                       # Activation
            nn.Dropout(0.5),                 # Dropout for regularization (50% probability) (Reference #4)
            # Dropout helps prevent overfitting by randomly setting a fraction of input units to 0 during training
                # Specifically, it prevents the neurons from relying too much on certain paths
            # This gets disabled during evaluation
            nn.Linear(512, num_classes)      # Final output layer for 52 classes (cards)
        )

    # Forward function derived from this article: (Reference #6) 
    # This defines how each node propogates its output to all the other nodes in the next layer
    def forward(self, x):
        # x: input tensor of shape [batch_size, 3, 256, 256]
        x = self.features(x)                   # Output shape: [batch_size, 256, 16, 16]
        # Flatten the feature maps into a single vector per image
        x = x.view(x.size(0), -1)             # Output shape: [batch_size, 256*16*16]
        # Pass through fully connected classifier
        x = self.classifier(x)                # Output shape: [batch_size, 52]

        return x

In [ ]:
# Use GPU or CPU for training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

# instantiate the model with 52 classes
net = SimpleCNN(num_classes=52).to(device)
# Define loss function
criterion = nn.CrossEntropyLoss()
# Define backpropogation function
optimizer = optim.Adam(net.parameters(), lr=0.001)

# Train on all the data 10 times
epochs = 10

# Get reference that shows roughly this pattern for traiing on pytorch docs
for epoch in range(epochs):
    print(f"\n----- Epoch {epoch+1}/{epochs} -----")
    # Switch the model to training mode
    net.train()
    # Define accumulators for running loss, # of correct predictions and total images trained
    train_loss = 0.0
    correct = 0
    total = 0
    for images, labels in trainloader:
        # trainloader will pair the images with their appropriate labels
        images, labels = images.to(device), labels.to(device)
        # Gradients for the backpropogation step are set to 0 when training
            # This ensures that any extra calcuations aren't executed (Reference #3)
        optimizer.zero_grad()
        # Pass training images to the CNN, which generates outputs as a tensor of one of the 52 classes
        outputs = net(images)
        # Use the predicted class and the image's label to generate loss via loss function
        loss = criterion(outputs, labels)
        # Update the gradients based on the computed loss
            # This effectively bridges the interaction between criterion and optimizer (loss calculation and backpropogation)
        loss.backward()
        # The optimizer backpropogates (updates weights per node) based on the new gradients
        optimizer.step()
        # Add the loss for this image to the accumulator (used to visualize progress)
        train_loss += loss.item()
        # Compress the output and turn it into one of the class numbers (1-52) (Reference #4)
        _, predicted = outputs.max(1)
        # Add the total number of labels from the trainloader (52)
        total += labels.size(0)
        # Count how many predictions were correct
        correct += predicted.eq(labels).sum().item()

    train_acc = 100 * correct / total
    print(f"Train Loss: {train_loss/len(trainloader):.4f} | Train Acc: {train_acc:.2f}%")
    # Switch the model to evaluation mode
    net.eval()
    # The validation loop uses similar concepts from the training loop
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    # The entire model is set to 0 gradient, since we don't need it to learn more at this point 
        # We instead need to test what it has learned up until now
    with torch.no_grad():
        for images, labels in valloader:
            images, labels = images.to(device), labels.to(device) 
            outputs = net(images)
            loss = criterion(outputs, labels)
            # Add validation loss to the accumulator 
            val_loss += loss.item()
            _, predicted = outputs.max(1) 
            val_total += labels.size(0) 
            val_correct += predicted.eq(labels).sum().item()

    val_acc = 100 * val_correct / val_total
    print(f"Val Loss: {val_loss/len(valloader):.4f} | Val Acc: {val_acc:.2f}%")

print("\nTraining Finished.")
torch.save(net.state_dict(), "card_model.pth")
print("Model saved as card_model.pth")


Using: cuda

----- Epoch 1/10 -----

----- Epoch 1/10 -----
Train Loss: 3.1355 | Train Acc: 19.22%
Train Loss: 3.1355 | Train Acc: 19.22%
Val Loss: 1.5129 | Val Acc: 57.31%

----- Epoch 2/10 -----
Val Loss: 1.5129 | Val Acc: 57.31%

----- Epoch 2/10 -----
Train Loss: 1.3717 | Train Acc: 57.36%
Train Loss: 1.3717 | Train Acc: 57.36%
Val Loss: 0.4940 | Val Acc: 84.81%

----- Epoch 3/10 -----
Val Loss: 0.4940 | Val Acc: 84.81%

----- Epoch 3/10 -----
Train Loss: 0.6894 | Train Acc: 78.68%
Train Loss: 0.6894 | Train Acc: 78.68%
Val Loss: 0.2103 | Val Acc: 94.04%

----- Epoch 4/10 -----
Val Loss: 0.2103 | Val Acc: 94.04%

----- Epoch 4/10 -----
Train Loss: 0.3999 | Train Acc: 87.60%
Train Loss: 0.3999 | Train Acc: 87.60%
Val Loss: 0.1169 | Val Acc: 96.73%

----- Epoch 5/10 -----
Val Loss: 0.1169 | Val Acc: 96.73%

----- Epoch 5/10 -----
Train Loss: 0.3178 | Train Acc: 90.36%
Train Loss: 0.3178 | Train Acc: 90.36%
Val Loss: 0.1091 | Val Acc: 96.92%

----- Epoch 6/10 -----
Val Loss: 0.1091 | 

In [ ]:
# ===================== REFERENCES ====================#
# Code/logic derived from references will be marked with (Reference #)
#
# 1. PyTorch DataLoader - https://docs.pytorch.org/docs/stable/data.html
# 2. PyTorch Module - https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module
# 3. Zeroing Gradients - https://docs.pytorch.org/tutorials/recipes/recipes/zeroing_out_gradients.html
# 4. ChatGPT (OpenAI)
# 5. Convolutional layer design - https://compsci682.github.io/notes/convolutional-networks/#:~:text=In%20this%20way%2C%20ConvNets%20transform,do%2C%20RELU%20doesn%27t)
# 6. Forward function design - https://medium.com/@myringoleMLGOD/simple-convolutional-neural-network-cnn-for-dummies-in-pytorch-a-step-by-step-guide-6f4109f6df80